In [1]:
using Turing, StatsPlots, Distributions, CSV, DataFrames, Optim

# Load the observed dataset
data = CSV.read("../../data/ice_cream_sales.csv", DataFrame);
names(data)

┌ Warning: The call to compilecache failed to create a usable precompiled cache file for NamedArrays [86f7a689-2022-50b4-a561-43c23ac3c673]
│   exception = ErrorException("Required dependency DelimitedFiles [8bb1440f-4735-579b-a4ab-409b98df4dab] failed to load from a cache file.")
└ @ Base loading.jl:1818


6-element Vector{String}:
 "ID"
 "Temperature"
 "Is_Weekend"
 "Hours_Open"
 "Electricity_Usage"
 "Ice_Cream_Sales"

In [2]:
describe(data)

Row,variable,mean,min,median,max,nmissing,eltype
,Symbol,Float64,Real,Float64,Real,Int64,DataType
1,ID,180.5,1,180.5,360,0,Int64
2,Temperature,24.8871,19.3161,24.8958,30.6386,0,Float64
3,Is_Weekend,0.283333,false,0.0,true,0,Bool
4,Hours_Open,9.53611,8,10.0,11,0,Int64
5,Electricity_Usage,87.7104,63.6465,87.7292,112.66,0,Float64
6,Ice_Cream_Sales,1034.57,813,1027.0,1276,0,Int64


In [3]:
#= Convert continous data to Float64
And the categorical data to integers =#

data[!, :Temperature] = convert(Array{Float64}, data[!, :Temperature])
data[!, :Is_Weekend] = convert(Array{Int}, data[!, :Is_Weekend])
data[!, :Hours_Open] = convert(Array{Int}, data[!, :Hours_Open])
data[!, :Ice_Cream_Sales] = convert(Array{Float64}, data[!, :Ice_Cream_Sales])
data[!, :Electricity_Usage] = convert(Array{Float64}, data[!, :Electricity_Usage]);

In [4]:
# Get variables
temperature = data[!, :Temperature]
is_weekend = data[!, :Is_Weekend]
hours_open = data[!, :Hours_Open]
electricity_usage = data[!, :Electricity_Usage]
ice_cream_sales = data[!, :Ice_Cream_Sales];

In [5]:
# Build the model

@model function ice_cream_sales_model(temperature, is_weekend, hours_open, electricity_usage, ice_cream_sales)
    n = min(length(temperature), length(is_weekend), length(hours_open), length(electricity_usage), length(ice_cream_sales))
    
    for i in 1:n
        # Priors for independent variables
        temperature[i] ~ Normal(25, 2)
        is_weekend[i] ~ Bernoulli(2/7)

        # How each model is dependent on it's predecessors
        if is_weekend[i] == 1
            hours_open[i] ~ DiscreteUniform(10,11)
        else
            hours_open[i] ~ DiscreteUniform(8,10)
        end

        electricity_usage[i] ~ Normal(10 + temperature[i] * 2 + hours_open[i] * 3, 1)
        
        # How the dependent variable is dependent on the independent variables
        ice_cream_sales[i] ~ Normal(20 + temperature[i] * 30 + hours_open[i] * 25 + is_weekend[i] * 100, 3)
    end
end

ice_cream_sales_model (generic function with 2 methods)

In [6]:
model = ice_cream_sales_model(repeat([missing], length(temperature)), repeat([missing], length(temperature)), repeat([missing], length(temperature)), repeat([missing], length(temperature)), repeat([missing], length(temperature)))

DynamicPPL.Model{typeof(ice_cream_sales_model), (:temperature, :is_weekend, :hours_open, :electricity_usage, :ice_cream_sales), (), (), NTuple{5, Vector{Missing}}, Tuple{}, DynamicPPL.DefaultContext}(ice_cream_sales_model, (temperature = [missing, missing, missing, missing, missing, missing, missing, missing, missing, missing  …  missing, missing, missing, missing, missing, missing, missing, missing, missing, missing], is_weekend = [missing, missing, missing, missing, missing, missing, missing, missing, missing, missing  …  missing, missing, missing, missing, missing, missing, missing, missing, missing, missing], hours_open = [missing, missing, missing, missing, missing, missing, missing, missing, missing, missing  …  missing, missing, missing, missing, missing, missing, missing, missing, missing, missing], electricity_usage = [missing, missing, missing, missing, missing, missing, missing, missing, missing, missing  …  missing, missing, missing, missing, missing, missing, missing, miss

In [7]:
cont_sampler = HMC(0.01, 10, :temperature, :electricity_usage, :ice_cream_sales)
disc_sampler = PG(10, :is_weekend, :hours_open)
sampler = Gibbs(cont_sampler, disc_sampler)

Gibbs{(:temperature, :electricity_usage, :ice_cream_sales, :is_weekend, :hours_open), Tuple{HMC{Turing.Essential.ForwardDiffAD{0}, (:temperature, :electricity_usage, :ice_cream_sales), AdvancedHMC.UnitEuclideanMetric}, PG{(:is_weekend, :hours_open), AdvancedPS.ResampleWithESSThreshold{typeof(AdvancedPS.resample_systematic), Float64}}}}((HMC{Turing.Essential.ForwardDiffAD{0}, (:temperature, :electricity_usage, :ice_cream_sales), AdvancedHMC.UnitEuclideanMetric}(0.01, 10), PG{(:is_weekend, :hours_open), AdvancedPS.ResampleWithESSThreshold{typeof(AdvancedPS.resample_systematic), Float64}}(10, AdvancedPS.ResampleWithESSThreshold{typeof(AdvancedPS.resample_systematic), Float64}(AdvancedPS.resample_systematic, 0.5))))

In [8]:
chain = sample(model, sampler, MCMCThreads(), 10, 6)

Sampling (6 threads)   0%|                              |  ETA: N/A
Sampling (6 threads)  17%|█████                         |  ETA: 0:09:42
Sampling (6 threads)  33%|██████████                    |  ETA: 0:03:53
Sampling (6 threads)  50%|███████████████               |  ETA: 0:01:56
Sampling (6 threads)  67%|████████████████████          |  ETA: 0:00:58
Sampling (6 threads)  83%|█████████████████████████     |  ETA: 0:00:23
Sampling (6 threads) 100%|██████████████████████████████| Time: 0:01:56
Sampling (6 threads) 100%|██████████████████████████████| Time: 0:01:56


Chains MCMC chain (10×1801×6 Array{Float64, 3}):

Iterations        = 1:1:10
Number of chains  = 6
Samples per chain = 10
Wall duration     = 85.17 seconds
Compute duration  = 507.88 seconds
parameters        = temperature[1], temperature[2], temperature[3], temperature[4], temperature[5], temperature[6], temperature[7], temperature[8], temperature[9], temperature[10], temperature[11], temperature[12], temperature[13], temperature[14], temperature[15], temperature[16], temperature[17], temperature[18], temperature[19], temperature[20], temperature[21], temperature[22], temperature[23], temperature[24], temperature[25], temperature[26], temperature[27], temperature[28], temperature[29], temperature[30], temperature[31], temperature[32], temperature[33], temperature[34], temperature[35], temperature[36], temperature[37], temperature[38], temperature[39], temperature[40], temperature[41], temperature[42], temperature[43], temperature[44], temperature[45], temperature[46], temperature[47],

In [9]:
model2 = ice_cream_sales_model(repeat([25], length(temperature)), is_weekend, hours_open, electricity_usage, repeat([missing], length(temperature)))

DynamicPPL.Model{typeof(ice_cream_sales_model), (:temperature, :is_weekend, :hours_open, :electricity_usage, :ice_cream_sales), (), (), Tuple{Vector{Int64}, Vector{Int64}, Vector{Int64}, Vector{Float64}, Vector{Missing}}, Tuple{}, DynamicPPL.DefaultContext}(ice_cream_sales_model, (temperature = [25, 25, 25, 25, 25, 25, 25, 25, 25, 25  …  25, 25, 25, 25, 25, 25, 25, 25, 25, 25], is_weekend = [0, 0, 0, 0, 0, 1, 1, 0, 0, 0  …  0, 0, 0, 0, 0, 1, 1, 0, 0, 0], hours_open = [10, 9, 10, 9, 9, 11, 11, 10, 10, 8  …  8, 9, 10, 8, 10, 11, 11, 10, 10, 10], electricity_usage = [79.9888221031828, 82.00205853574766, 82.20962512128071, 94.70264318970042, 68.83438624966436, 78.06102934854225, 98.4275988256301, 85.03319420107236, 78.54189279419404, 79.56872837110795  …  83.13639034552708, 90.22215868118067, 89.72708966445728, 77.46861446025392, 91.39167865839339, 96.84577176795625, 66.93461618133804, 99.87608048519269, 85.47904611441922, 85.4934011872982], ice_cream_sales = [missing, missing, missing, mi

In [10]:
chain_counterfactual = sample(model2, sampler, 10)

Sampling   0%|                                          |  ETA: N/A
Sampling  10%|████▎                                     |  ETA: 0:00:56
Sampling  20%|████████▍                                 |  ETA: 0:00:47
Sampling  30%|████████████▋                             |  ETA: 0:00:34
Sampling  40%|████████████████▊                         |  ETA: 0:00:25
Sampling  50%|█████████████████████                     |  ETA: 0:00:19
Sampling  60%|█████████████████████████▎                |  ETA: 0:00:15
Sampling  70%|█████████████████████████████▍            |  ETA: 0:00:10
Sampling  80%|█████████████████████████████████▋        |  ETA: 0:00:07
Sampling  90%|█████████████████████████████████████▊    |  ETA: 0:00:03
Sampling 100%|██████████████████████████████████████████| Time: 0:00:31
Sampling 100%|██████████████████████████████████████████| Time: 0:00:31


Chains MCMC chain (10×361×1 Array{Float64, 3}):

Iterations        = 1:1:10
Number of chains  = 1
Samples per chain = 10
Wall duration     = 31.99 seconds
Compute duration  = 31.99 seconds
parameters        = ice_cream_sales[1], ice_cream_sales[2], ice_cream_sales[3], ice_cream_sales[4], ice_cream_sales[5], ice_cream_sales[6], ice_cream_sales[7], ice_cream_sales[8], ice_cream_sales[9], ice_cream_sales[10], ice_cream_sales[11], ice_cream_sales[12], ice_cream_sales[13], ice_cream_sales[14], ice_cream_sales[15], ice_cream_sales[16], ice_cream_sales[17], ice_cream_sales[18], ice_cream_sales[19], ice_cream_sales[20], ice_cream_sales[21], ice_cream_sales[22], ice_cream_sales[23], ice_cream_sales[24], ice_cream_sales[25], ice_cream_sales[26], ice_cream_sales[27], ice_cream_sales[28], ice_cream_sales[29], ice_cream_sales[30], ice_cream_sales[31], ice_cream_sales[32], ice_cream_sales[33], ice_cream_sales[34], ice_cream_sales[35], ice_cream_sales[36], ice_cream_sales[37], ice_cream_sales[38], ic

In [11]:
model3 = ice_cream_sales_model(temperature, is_weekend, repeat([10], length(temperature)), electricity_usage, repeat([missing], length(temperature)))

DynamicPPL.Model{typeof(ice_cream_sales_model), (:temperature, :is_weekend, :hours_open, :electricity_usage, :ice_cream_sales), (), (), Tuple{Vector{Float64}, Vector{Int64}, Vector{Int64}, Vector{Float64}, Vector{Missing}}, Tuple{}, DynamicPPL.DefaultContext}(ice_cream_sales_model, (temperature = [26.616575856929934, 22.755854983771652, 22.790727795341407, 24.166014729670135, 25.575175961247712, 25.459637396103734, 24.156462671200615, 22.288818757797607, 25.138918282183788, 24.765354390938374  …  29.0533449321096, 23.42193290462059, 24.512600942879118, 24.03646367794297, 23.868907312282715, 24.33236353944156, 21.105267862807114, 26.540918408243332, 23.3604840358554, 23.694850507188377], is_weekend = [0, 0, 0, 0, 0, 1, 1, 0, 0, 0  …  0, 0, 0, 0, 0, 1, 1, 0, 0, 0], hours_open = [10, 10, 10, 10, 10, 10, 10, 10, 10, 10  …  10, 10, 10, 10, 10, 10, 10, 10, 10, 10], electricity_usage = [79.9888221031828, 82.00205853574766, 82.20962512128071, 94.70264318970042, 68.83438624966436, 78.0610293485

In [12]:
chain_intervention = sample(model3, sampler, 10)

Sampling   0%|                                          |  ETA: N/A
Sampling  10%|████▎                                     |  ETA: 0:00:49
Sampling  20%|████████▍                                 |  ETA: 0:00:43
Sampling  30%|████████████▋                             |  ETA: 0:00:31
Sampling  40%|████████████████▊                         |  ETA: 0:00:25
Sampling  50%|█████████████████████                     |  ETA: 0:00:19
Sampling  60%|█████████████████████████▎                |  ETA: 0:00:15
Sampling  70%|█████████████████████████████▍            |  ETA: 0:00:10
Sampling  80%|█████████████████████████████████▋        |  ETA: 0:00:07
Sampling  90%|█████████████████████████████████████▊    |  ETA: 0:00:03
Sampling 100%|██████████████████████████████████████████| Time: 0:00:32
Sampling 100%|██████████████████████████████████████████| Time: 0:00:32


Chains MCMC chain (10×361×1 Array{Float64, 3}):

Iterations        = 1:1:10
Number of chains  = 1
Samples per chain = 10
Wall duration     = 32.41 seconds
Compute duration  = 32.41 seconds
parameters        = ice_cream_sales[1], ice_cream_sales[2], ice_cream_sales[3], ice_cream_sales[4], ice_cream_sales[5], ice_cream_sales[6], ice_cream_sales[7], ice_cream_sales[8], ice_cream_sales[9], ice_cream_sales[10], ice_cream_sales[11], ice_cream_sales[12], ice_cream_sales[13], ice_cream_sales[14], ice_cream_sales[15], ice_cream_sales[16], ice_cream_sales[17], ice_cream_sales[18], ice_cream_sales[19], ice_cream_sales[20], ice_cream_sales[21], ice_cream_sales[22], ice_cream_sales[23], ice_cream_sales[24], ice_cream_sales[25], ice_cream_sales[26], ice_cream_sales[27], ice_cream_sales[28], ice_cream_sales[29], ice_cream_sales[30], ice_cream_sales[31], ice_cream_sales[32], ice_cream_sales[33], ice_cream_sales[34], ice_cream_sales[35], ice_cream_sales[36], ice_cream_sales[37], ice_cream_sales[38], ic

In [13]:
model4 = ice_cream_sales_model([25], [1], [11], [80], [missing])

DynamicPPL.Model{typeof(ice_cream_sales_model), (:temperature, :is_weekend, :hours_open, :electricity_usage, :ice_cream_sales), (), (), Tuple{Vector{Int64}, Vector{Int64}, Vector{Int64}, Vector{Int64}, Vector{Missing}}, Tuple{}, DynamicPPL.DefaultContext}(ice_cream_sales_model, (temperature = [25], is_weekend = [1], hours_open = [11], electricity_usage = [80], ice_cream_sales = [missing]), NamedTuple(), DynamicPPL.DefaultContext())

In [15]:
chain_singular = sample(model4, sampler, 1000)

Sampling   0%|                                          |  ETA: N/A
Sampling   0%|▎                                         |  ETA: 0:00:55
Sampling   1%|▍                                         |  ETA: 0:00:53
Sampling   2%|▋                                         |  ETA: 0:00:52
Sampling   2%|▉                                         |  ETA: 0:00:53
Sampling   2%|█                                         |  ETA: 0:00:54
Sampling   3%|█▎                                        |  ETA: 0:00:54
Sampling   4%|█▌                                        |  ETA: 0:00:54
Sampling   4%|█▋                                        |  ETA: 0:00:54
Sampling   4%|█▉                                        |  ETA: 0:00:53
Sampling   5%|██▏                                       |  ETA: 0:00:53
Sampling   6%|██▎                                       |  ETA: 0:00:53
Sampling   6%|██▌                                       |  ETA: 0:00:52
Sampling   6%|██▊                                       |  ETA: 0:00

Chains MCMC chain (1000×2×1 Array{Float64, 3}):

Iterations        = 1:1:1000
Number of chains  = 1
Samples per chain = 1000
Wall duration     = 58.46 seconds
Compute duration  = 58.46 seconds
parameters        = ice_cream_sales[1]
internals         = lp

Summary Statistics
          parameters        mean       std   naive_se      mcse       ess      ⋯
              Symbol     Float64   Float64    Float64   Float64   Float64   Fl ⋯

  ice_cream_sales[1]   1147.6343    1.2244     0.0387    0.2173    2.4059    2 ⋯
                                                               2 columns omitted

Quantiles
          parameters        2.5%       25.0%       50.0%       75.0%       97. ⋯
              Symbol     Float64     Float64     Float64     Float64     Float ⋯

  ice_cream_sales[1]   1145.5941   1146.5179   1147.6145   1148.7988   1149.52 ⋯
                                                                1 column omitted
